# dbAppsUA09: Безпека та транзакції
## Покроковий довідник

У цьому уроці:
- **SQL ін'єкція (SQL Injection)** — як зловмисники атакують бази даних
- **Параметризовані запити** — як захиститися
- **Транзакції** — як забезпечити цілісність даних
- **ACID** — властивості надійних транзакцій

## Налаштування

In [ ]:
import pandas as pd
import sqlite3

# Створюємо базу даних в пам'яті для демонстрації
conn = sqlite3.connect(":memory:")

print("Підключення встановлено.")

In [ ]:
# Створюємо таблицю користувачів з тестовими даними
conn.execute("""
CREATE TABLE users (
    userId INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT NOT NULL,
    password TEXT NOT NULL,
    email TEXT NOT NULL
)
""")

sampleUsers = [
    ("admin", "password123", "admin@company.com"),
    ("alice", "secret456", "alice@company.com"),
    ("bob", "bobpass789", "bob@company.com"),
]

conn.executemany("INSERT INTO users (username, password, email) VALUES (?, ?, ?)", sampleUsers)
conn.commit()

dfUsers = pd.read_sql("SELECT * FROM users", conn)
print(dfUsers)

## Частина 1: SQL Ін'єкція (SQL Injection)

SQL ін'єкція — це атака де зловмисний SQL код вставляється через поле введення користувача.

In [ ]:
# ВРАЗЛИВИЙ підхід: конкатенація рядків
# НЕ використовуйте це в реальних проектах!

userInput = "admin"
vulnerableQuery = f"SELECT * FROM users WHERE username = '{userInput}'"

print("Запит:")
print(vulnerableQuery)
print()

result = pd.read_sql(vulnerableQuery, conn)
print("Результат з userInput = 'admin':")
print(result)

### Проблема: Зловмисний ввід

Якщо користувач введе `' OR '1'='1`, запит стає:

```sql
SELECT * FROM users WHERE username = '' OR '1'='1'
```

Умова `'1'='1'` **завжди True** → повертає **ВСІХ** користувачів! Авторизація обійдена.

## БЕЗПЕЧНИЙ підхід: Параметризовані запити

Використовуйте `?` як заповнювач. База даних обробляє значення як **ДАНІ**, а не як SQL код.

In [ ]:
# БЕЗПЕЧНИЙ підхід: параметризований запит з ?

userInput = "admin"
safeQuery = "SELECT * FROM users WHERE username = ?"

result = pd.read_sql(safeQuery, conn, params=(userInput,))
print("Безпечний результат:")
print(result)

In [ ]:
# Спроба атаки через безпечний запит
maliciousInput = "' OR '1'='1"
safeQuery = "SELECT * FROM users WHERE username = ?"

result = pd.read_sql(safeQuery, conn, params=(maliciousInput,))
print(f"Результат з зловмисним вводом:")
print(result)
print()
print("Жодних рядків! Ввід трактується як звичайний текст, не SQL код.")

## Частина 2: Транзакції (Transactions)

**Транзакція** — це група SQL операцій які повинні **всі виконатися або всі скасуватися**. Все або нічого.

| Команда | Що робить |
|---------|----------|
| **BEGIN** | Починає транзакцію |
| **COMMIT** | Зберігає всі зміни |
| **ROLLBACK** | Скасовує всі зміни |

In [ ]:
# Створюємо таблицю банківських рахунків
conn.execute("""
CREATE TABLE bank_accounts (
    accountId INTEGER PRIMARY KEY AUTOINCREMENT,
    accountName TEXT NOT NULL,
    balance REAL NOT NULL
)
""")

accounts = [("Alice", 1000.00), ("Bob", 5000.00), ("Charlie", 500.00)]
conn.executemany("INSERT INTO bank_accounts (accountName, balance) VALUES (?, ?)", accounts)
conn.commit()

print(pd.read_sql("SELECT * FROM bank_accounts", conn))

In [ ]:
# Приклад COMMIT: Переказ $100 від Alice до Bob
conn.execute("BEGIN")
conn.execute("UPDATE bank_accounts SET balance = balance - 100 WHERE accountId = 1")
conn.execute("UPDATE bank_accounts SET balance = balance + 100 WHERE accountId = 2")
conn.commit()

print("Транзакцію збережено (COMMIT). Гроші переказано.")
print()
print(pd.read_sql("SELECT * FROM bank_accounts", conn))

In [ ]:
# Приклад ROLLBACK: Charlie намагається переказати $1000 (має тільки $500)
conn.execute("BEGIN")
conn.execute("UPDATE bank_accounts SET balance = balance - 1000 WHERE accountId = 3")

# Перевіряємо баланс
check = pd.read_sql("SELECT balance FROM bank_accounts WHERE accountId = 3", conn)
print(f"Баланс Charlie після зняття: ${check['balance'][0]}")
print("ПРОБЛЕМА: Баланс від'ємний!")
print()

# Скасовуємо транзакцію
conn.execute("ROLLBACK")
print("Транзакцію скасовано (ROLLBACK).")

# Перевіряємо що баланс відновлено
check = pd.read_sql("SELECT * FROM bank_accounts WHERE accountId = 3", conn)
print("Рахунок Charlie після ROLLBACK:")
print(check)

## ACID — Властивості надійних транзакцій

| Літера | Властивість | Пояснення |
|--------|-------------|----------|
| **A** | Atomicity (Атомарність) | Всі кроки виконуються, або жоден |
| **C** | Consistency (Узгодженість) | БД переходить з одного правильного стану в інший |
| **I** | Isolation (Ізоляція) | Транзакції не заважають одна одній |
| **D** | Durability (Стійкість) | Після COMMIT зміни збережено назавжди |

## Захист даних

Бази даних часто містять чутливу інформацію. Розробники повинні:
1. Знати які чутливі дані зберігаються
2. Контролювати хто має доступ
3. Використовувати параметризовані запити
4. Використовувати транзакції для цілісності

**Спробуй:** Напиши транзакцію що переказує $200 від Bob (accountId 2) до Charlie (accountId 3). Включи BEGIN, два UPDATE, COMMIT, та SELECT для перевірки.

In [ ]:
# Спробуй: Транзакція — переказ $200 від Bob до Charlie

## Підсумок

1. **Параметризовані запити** запобігають SQL ін'єкціям — ніколи не конкатенуйте!
2. **Транзакції** гарантують що багатокрокові операції виконуються повністю або скасовуються
3. **BEGIN**, **COMMIT**, **ROLLBACK** — три основні команди
4. **ACID** — обіцянка надійності: Атомарність, Узгодженість, Ізоляція, Стійкість